In [13]:
import requests
import numpy as np
import pandas as pd
import sqlite3
from datetime import datetime, timezone


Create tables

In [14]:
conn = sqlite3.connect("screener.db")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS pools (
    pair_address TEXT PRIMARY KEY,
    token_address TEXT,
    chain_id TEXT,
    symbol TEXT,
    dex_id TEXT,
    price_usd REAL,
    liquidity_usd REAL,
    volume_m5 REAL,
    volume_h1 REAL,
    volume_h24 REAL,
    price_change_m5 REAL,
    price_change_h1 REAL,
    price_change_h24 REAL,
    market_cap REAL,
    fdv REAL,
    pair_created_at INTEGER,
    first_seen_at TIMESTAMP,
    last_updated_at TIMESTAMP
)
""")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS pool_snapshots (
        id               INTEGER PRIMARY KEY AUTOINCREMENT,
        pair_address     TEXT REFERENCES pools(pair_address),
        price_usd        REAL,
        liquidity_usd    REAL,
        volume_m5        REAL,
        volume_h1        REAL,
        volume_h24       REAL,
        price_change_m5  REAL,
        price_change_h1  REAL,
        price_change_h24 REAL,
        market_cap       REAL,
        fdv              REAL,
        snapshot_at      TIMESTAMP
    )
""")

In [15]:
conn.commit()

response = requests.get("https://api.dexscreener.com/token-profiles/latest/v1", headers={"Accept": "*/*"})
tokens = response.json()
all_pairs = []
new_tokens_count = 0

for token in tokens:
    chain_id = token.get("chainId")
    token_address = token.get("tokenAddress")

    cursor.execute(
        """SELECT 1 FROM pools WHERE token_address = ? """,
        (token_address,)
    )

    if cursor.fetchone():
        continue

    print(f"New token discovered: {token_address}")
    new_tokens_count += 1

    now = datetime.now(timezone.utc).isoformat()
    data = requests.get(f"https://api.dexscreener.com/token-pairs/v1/{chain_id}/{token_address}",headers={"Accept": "*/*"})
    pair_response = data.json()

    pair_details = [{ "token_address" : token_address, 
                     "symbol" : item.get('baseToken').get('symbol'),
                     "pair_address" : item.get('pairAddress'),
                     "dex_id" : item.get('dexId'),
                     "price_usd" : item.get('priceUsd'),
                     "price_change_m5" : item.get('priceChange', {}).get('m5'),
                     "price_change_h1" : item.get('priceChange', {}).get('h1'),
                     "price_change_h24" : item.get('priceChange', {}).get('h24'),
                     "liquidity_usd" : item.get('liquidity', {}).get('usd'),
                     "volume_m5" : item.get('volume', {}).get('m5'),
                     "volume_h1" : item.get('volume', {}).get('h1'),
                     "volume_h24" : item.get('volume', {}).get('h24'),
                     "market_cap" : item.get('marketCap'),
                     "fdv" : item.get('fdv'),
                     "pair_created_at" : item.get('pairCreatedAt'),
                     "timestamp_fetched" : now,
    }
    for item in pair_response ]
    all_pairs.extend(pair_details)

    for pool in pair_details:
        cursor.execute(
            """
            INSERT INTO pools (
                pair_address,
                token_address,
                chain_id,
                symbol,
                dex_id,
                price_usd,
                liquidity_usd,
                volume_m5,
                volume_h1,
                volume_h24,
                price_change_m5,
                price_change_h1,
                price_change_h24,
                market_cap,
                fdv,
                pair_created_at,
                first_seen_at,
                last_updated_at
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?) ON CONFLICT(pair_address) DO UPDATE SET
                price_usd       = excluded.price_usd,
                liquidity_usd   = excluded.liquidity_usd,
                volume_m5       = excluded.volume_m5,
                volume_h1       = excluded.volume_h1,
                volume_h24      = excluded.volume_h24,
                price_change_m5  = excluded.price_change_m5,
                price_change_h1  = excluded.price_change_h1,
                price_change_h24 = excluded.price_change_h24,
                market_cap      = excluded.market_cap,
                fdv             = excluded.fdv,
                last_updated_at = excluded.last_updated_at
            """,
            (
                pool["pair_address"],
                pool["token_address"],
                chain_id,
                pool["symbol"],
                pool["dex_id"],
                pool["price_usd"],
                pool["liquidity_usd"],
                pool["volume_m5"],
                pool["volume_h1"],
                pool["volume_h24"],
                pool["price_change_m5"],
                pool["price_change_h1"],
                pool["price_change_h24"],
                pool["market_cap"],
                pool["fdv"],
                pool["pair_created_at"],
                now,
                now
            )
        )
        cursor.execute(
            """
            INSERT INTO pool_snapshots (
                pair_address, 
                price_usd, 
                liquidity_usd,
                volume_m5, 
                volume_h1, 
                volume_h24,
                price_change_m5, 
                price_change_h1, 
                price_change_h24,
                market_cap, 
                fdv, snapshot_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                pool["pair_address"], 
                pool["price_usd"], 
                pool["liquidity_usd"],
                pool["volume_m5"], 
                pool["volume_h1"], 
                pool["volume_h24"],
                pool["price_change_m5"], 
                pool["price_change_h1"], 
                pool["price_change_h24"],
                pool["market_cap"], 
                pool["fdv"], now
            )
)      
conn.commit()

df = pd.DataFrame(all_pairs)
print(f"New tokens discovered: {new_tokens_count}")
print(f"Pairs collected: {len(df)}")
conn.close()

New token discovered: 45a1fk7ygjqjRpEGVxL2v5em4TrdjZ121KVCznZUWNDY
New token discovered: 4rbSDqHYv24sLDYkxJJBbUZyLM56dto6prJyihTnpump
New token discovered: 4AKE1e6j22UqGroA36pNkMNFVBsezehMT4XKik7qmwbA
New token discovered: 5xHMRXNcrsipK89EGPN8whB38DWCwLSmPdwG6TQ8pump
New token discovered: DiceZJNFSUJvHHHJCXTeQ7eLTUXzQb9Fq7EB6EKZ3Ro5
New token discovered: 2Ti2QncSxY28gvuPmT9xwem8Xm9ASaGhEcm4BLEapump
New token discovered: GzTRWpK5tRBeujE4BoGjrW8f3uessXcHxDKntcsppump
New token discovered: 9ZXzELawSgtpKPwQEd9z4iEhPhKGEUWUWa671fVqpump
New token discovered: 0xe26b7183B9Ed8d596Dcac5ee1B319E8817454F5b
New token discovered: 4emP879oQe5F72ugd5GfqkV2q7j15K8XQgJfDpnCpump
New token discovered: 8ZqFyHJpAVJoAPWxyaMAu8d18a7U6Z8PhGTH99Zmpump
New token discovered: HG5aAuCVdVNEtL6GsvsEu5sxZkK9kT9pqG47dxRHpump
New token discovered: 81WAh3VXDTaN82traXP5ys7JNNtBayNXHtfsGGv2pump
New token discovered: Esycn8mW3K9yqQdx7AazTqSGNokXnDVAgfM5HsXqpump
New token discovered: 9mMTGZEqRtTcYDGm7KQUH5gmZEbWZeMsJkcCRShNpu

In [16]:
%reload_ext sql
%sql sqlite:////Users/ismail/GitHub_Projects/Solana-dexscreener_bot/screener.db

In [17]:
%%sql
SELECT pair_address,
       COUNT(*)
FROM pool_snapshots
GROUP BY pair_address
ORDER BY COUNT(*) DESC;

Running query in 'sqlite:////Users/ismail/GitHub_Projects/Solana-dexscreener_bot/screener.db'

pair_address,COUNT(*)
0x1Df1a345BE822B3fe07F7c412F9B12B9A0ecB26e,1
0x427bd05580b68700fd92296764ce9f6f856711c4ddd016787e309c3e437286d7,1
0xB5e1c3f00f97c21f55Aa5641e4264d1E77c9FFfF:4meme,1
14LPfwNLPpbBhjXsoEf2t8hKoT7V3kLDiMYE9KBvw4ey,1
2CgnRF2UdvDtCSL5Dp5VjwNPW6AchToMh5EM4P1dAjhM,1
2PBwRFq5omijBHpaarY2LENJfqe17v2tbZt7FZiUjee2,1
2V52moQzRpk5FGuNP4nXdxPxGLBLKGbpy2fN6FeS2965,1
2m6y4ZM3E8wCyRMpH3KE2VQqy4Dbjv7yc9UK5dsoxtMr,1
2oqH85YB61r3cSUdvgKdMRdY9uP5A4XMbbQFrEWSqKeU,1
33954mgGL4Hi83STSEmrgcxage7ZbBJFPjfCYfLcyqgd,1
